# Configuration

In [2]:
# * IMPORTS
from pathlib import Path
from typing import Dict, Final
from box import Box
import pandas as pd
import os
from IPython.display import display
from pprint import pprint
import plotly.express as px
from contextlib import redirect_stdout
import mne
import sys

WD = Path.cwd()
sys.path.append(WD)
os.chdir(WD)
assert WD == Path.cwd()

# * RELATIVE IMPORTS
# from analysis_conf import Config as c
# from data_loader.human_data import HumanSessData, HumanSubjData, HumanGroupData
# from utils.analysis_utils import read_file, list_contents
# from analysis_compare_clean import CombinedData
from abstract_reasoning_analysis.data_loader.human_data import (
    HumanSessData,
    HumanSubjData,
    HumanGroupData,
)
from abstract_reasoning_analysis.utils.custom_type_hints import DATA_FMTS
from abstract_reasoning_analysis.utils.analysis_utils import (
    read_file,
    reorder_item_ids,
    list_contents,
)
from abstract_reasoning_analysis.analysis_rsa import get_ds_and_rdm
from abstract_reasoning_analysis.analysis_config import Config as c

# ! TEMP: to locate and use ffmpeg
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ["PATH"]

In [5]:
# * GLOBAL VARIABLES
PATTERNS = c.PATTERNS
ANN_ID_MAPPING = c.ANN_ID_MAPPING
ANN_ID_ORDER = c.ANN_ID_ORDER

DATASET = c.DATASET
SEQ_FILE = c.SEQ_FILE
# DIRECTORIES = c.DIRECTORIES

# SAVE_DISK: Final = Path("/Volumes/Realtek 1Tb")
SAVE_DISK: Final = Path("/Volumes/SSD-512Go")
assert SAVE_DISK.exists(), "WARNING: SSD not connected"
MAIN_DATA_DIR = SAVE_DISK / "PhD Data/experiment1/data/"
DIRECTORIES: Final = Box(
    {
        "ann": {
            "data": MAIN_DATA_DIR
            / "ANNs/local_run/sessions-1_to_5-masked_idx(7)-sequence_tokens_acts",
            "prepro": None,
            "analyzed": MAIN_DATA_DIR / "ANNs/analyzed",
            "export": MAIN_DATA_DIR / "ANNs/analyzed",
        },
        "human": {
            "data": MAIN_DATA_DIR / "Lab/raw",
            "prepro": MAIN_DATA_DIR / "Lab/preprocessed",
            "analyzed": MAIN_DATA_DIR / "Lab/analyzed",
            "export": MAIN_DATA_DIR / "Lab/analyzed",
        },
    }
)


# Human Data

### Load Data

In [6]:
this_sess = HumanSessData(
    DIRECTORIES.human.data,
    DIRECTORIES.human.prepro,
    DIRECTORIES.human.export,
    subj_N=1,
    sess_N = 1
)

this_subj = HumanSubjData(
    DIRECTORIES.human.data,
    DIRECTORIES.human.prepro,
    DIRECTORIES.human.export,
    subj_N=1,
)

# this_sess = this_subj.sessions[1]

human_group = HumanGroupData(
    DIRECTORIES.human.data,
    DIRECTORIES.human.prepro,
    DIRECTORIES.human.export,
)


In [7]:
human_group.show_dir_struct()

Human Data Root Directory Structure:
raw/
├── dataset_description.json
├── participants.json
├── participants.tsv
├── README
├── sub-01/
│   ├── ses-01/
│   │   ├── beh/
│   │   │   ├── sub-01_ses-01_task-AbsPattComp_beh.json
│   │   │   └── sub-01_ses-01_task-AbsPattComp_beh.tsv
│   │   ├── eeg/
│   │   │   ├── sub-01_ses-01_task-AbsPattComp_channels.tsv
│   │   │   ├── sub-01_ses-01_task-AbsPattComp_eeg.bdf
│   │   │   ├── sub-01_ses-01_task-AbsPattComp_eeg.json
│   │   │   ├── sub-01_ses-01_task-AbsPattComp_events.json
│   │   │   └── sub-01_ses-01_task-AbsPattComp_events.tsv
│   │   ├── func/
│   │   │   ├── sub-01_ses-01_task-AbsPattComp_recording-eye1_physio.json
│   │   │   ├── sub-01_ses-01_task-AbsPattComp_recording-eye1_physio.tsv.gz
│   │   │   ├── sub-01_ses-01_task-AbsPattComp_recording-eye1_physioevents.json
│   │   │   └── sub-01_ses-01_task-AbsPattComp_recording-eye1_physioevents.tsv.gz
│   │   └── sub-01_ses-01_scans.tsv
│   ├── ses-02/
│   │   ├── beh/
│   │   │   ├──

In [37]:
list(this_sess.get_raw_behav_data().columns)

['subj_id',
 'trial_type',
 'item_id',
 'trial_onset_time',
 'series_end_time',
 'choice_onset_time',
 'rt',
 'rt_global',
 'choice_key',
 'solution_key',
 'choice',
 'solution',
 'correct',
 'pattern',
 'blockN',
 'iti']

In [38]:
sess_info = human_group.get_sess_info()

In [39]:
sess_info



{1: {1: {'session_id': 'ses-01',
   'vision_correction': 'contacts',
   'eye': 'left',
   'eye_screen_dist': 610,
   'window_size': '2560, 1440',
   'img_size': '256, 256',
   'notes': ''},
  2: {'session_id': 'ses-02',
   'vision_correction': 'contacts',
   'eye': 'left',
   'eye_screen_dist': 620,
   'window_size': '2560, 1440',
   'img_size': '256, 256',
   'notes': ''},
  3: {'session_id': 'ses-03',
   'vision_correction': 'contacts',
   'eye': 'left',
   'eye_screen_dist': 610,
   'window_size': '2560, 1440',
   'img_size': '256, 256',
   'notes': ''},
  4: {'session_id': 'ses-04',
   'vision_correction': 'glasses',
   'eye': 'left',
   'eye_screen_dist': 640,
   'window_size': '2560, 1440',
   'img_size': '256, 256',
   'notes': ''},
  5: {'session_id': 'ses-05',
   'vision_correction': 'glasses',
   'eye': 'left',
   'eye_screen_dist': 640,
   'window_size': '2560, 1440',
   'img_size': '256, 256',
   'notes': 'eye to screen distance was on cm, so it has now been corrected to mm

In [40]:
_df_sess_info = []
for subj_N, subj_info in sess_info.items():
    for sess_N, sessions_info in subj_info.items():
        _df = pd.DataFrame.from_dict(sessions_info, orient='index').T
        _df['subj_N'] = subj_N
        _df["sess_N"] = sess_N
        _df_sess_info.append(_df)
        
df_sess_info = pd.concat(_df_sess_info).reset_index(drop=True)

print(f"{df_sess_info['img_size'].unique() = }")
print(f"{df_sess_info['window_size'].unique() = }")
print(f"{df_sess_info['eye'].unique() = }")
print(f"{df_sess_info['vision_correction'].unique() = }")


df_sess_info['img_size'].unique() = array(['256, 256'], dtype=object)
df_sess_info['window_size'].unique() = array(['2560, 1440'], dtype=object)
df_sess_info['eye'].unique() = array(['left', 'right'], dtype=object)
df_sess_info['vision_correction'].unique() = array(['contacts', 'glasses', 'none'], dtype=object)


In [41]:
physio_jsons = [read_file(f) for f in list_contents(human_group.data_dir, reg=r".+_physio.json$")]
physio_jsons

[{'Manufacturer': 'SR-Research',
  'PhysioType': 'eyetrack',
  'Columns': ['timestamp', 'x_coordinate', 'y_coordinate', 'pupil_size'],
  'timestamp': {'Description': 'a continuously increasing identifier of the sampling time registered by the device',
   'Units': 'ms',
   'Origin': 'System startup'},
  'x_coordinate': {'LongName': 'Gaze position (x)',
   'Description': 'Gaze position x-coordinate of the recorded eye, in the coordinate units specified in the corresponding metadata sidecar.',
   'Units': 'pixel'},
  'y_coordinate': {'LongName': 'Gaze position (y)',
   'Description': 'Gaze position y-coordinate of the recorded eye, in the coordinate units specified in the corresponding metadata sidecar.',
   'Units': 'pixel'},
  'pupil_size': {'Description': "Pupil area of the recorded eye as calculated by the eye-tracker in arbitrary units (see EyeLink's documentation for conversion).",
   'Units': 'a.u.'},
  'SoftwareVersion': 'EYELINK II CL v5.50 Jun 16 2022 (EyeLink 1000 Plus)',
  'Sa

In [ ]:
ks = ["AverageCalibrationError", "MaximalCalibrationError"]
res = []
for pj in physio_jsons:
    for k in ks:
        res.extend(pj[k][0])

{<class 'float'>}


### Analyze Data

####  Subject Level

In [ ]:
this_subj.show_dir_struct()

Now let's look at this subject's performance

In [ ]:
this_subj_behav = this_subj.get_behav_data()
display(this_subj_behav.head())

agg_level = "session"  # "subject" or "session"
this_subj_perf_res = this_subj.analyze_perf(agg="subject")
perf_res_keys = this_subj_perf_res.keys()

print(f"Perfomance results' keys:{''.join([f'\n  - {k}' for k in perf_res_keys])}")

display(this_subj_perf_res["acc_by_patt"].head())


##### Create trial video

In [ ]:
# # * Load the data & Split it into trials
sess_info, behav, eeg, et = this_sess.get_data()

(
    manual_et_trials,
    *_,
    # et_events_dict,
    # et_events_dict_inv,
    # et_trial_bounds,
    # et_trial_events_df,
) = this_sess.split_et_data_into_trials(et)

(
    manual_eeg_trials,
    *_,
    # eeg_trial_bounds,
    # eeg_events,
    # eeg_events_df,
) = this_sess.split_eeg_data_into_trials(eeg, behav)


manual_et_trials = list(manual_et_trials)
manual_eeg_trials = list(manual_eeg_trials)

# * Generate a video for one trial
trial_N = 30
save_dir = Path(
    f"./TEST/subj{this_sess.subj_N:02}_ses{this_sess.sess_N:02}/trial{trial_N}_frames"
)
save_dir.mkdir(exist_ok=True, parents=True)

this_sess.generate_trial_video(
    manual_et_trials,
    manual_eeg_trials,
    behav,
    trial_N,
    # all_bad_chans: Optional[Dict] = None,
    # eeg_chan_groups: Optional[Dict] = None,
    # non_eeg_chans: Optional[List[str]] = None,
    # et_sfreq: Union[int, float, NoneType] = None,
    # valid_events_inv: Optional[Dict] = None,
    # eeg_sfreq: Union[int, float, NoneType] = None,
    # screen_resolution: Optional[Tuple[int, int]] = None,
    # icon_images: Optional[Dict[str, numpy.ndarray]] = None,
    save_dir=save_dir,
    gaze_pt_size=10,
    # gaze_ln_size = None
)


##### Sesssion Data

Let's get locate in which trials and in which presentation order each stimulus is shown

In [ ]:
stim_flash_order = this_subj.get_stim_flash_order()
display(stim_flash_order.dropna())

In [ ]:
ind = 0
row = stim_flash_order.dropna().iloc[ind : ind + 1]
display(row)

row = Box(row.iloc[0].to_dict())
# trial_N = row.overall_trial_N
trial_N = row.trial_N
stim = row.stim

print(
    f"""
`{row.stim}` is present in trial {row.trial_N} of session {row.sess_N} for subject
{row.subj_N} and is flashed at time slot(s) {row.stim_flash_order} of the encoding phase
""".replace("\n", " ").strip()
)

seq = ["[]" if i not in row.stim_locs else row.stim for i in range(8)]
options = ["[]" if i not in row.stim_locs else row.stim for i in range(8, 13)]

print(
    f"\nSpatial arrangment of this stimulus in the trial:\n\tSequence: {' '.join(seq)}\n\tOptions: {' '.join(options)}"
)


In [ ]:
fig_cols = [f"figure{i}" for i in range(1, 9)]
choice_cols = [f"choice{i}" for i in range(1, 5)]

trial_data = sequence = this_subj_behav.query(
    f"sess_N == {row['sess_N']} & trial_N=={row['trial_N']}"
)
trial_sequence = trial_data[fig_cols]
trial_choices = trial_data[choice_cols]

print(f"Sequence: {' '.join(trial_sequence.values[0].tolist())}")
print(f"Options: {' '.join(trial_choices.values[0].tolist())}")

Here's the average EEG activity for that stimulus across all trials of this session

In [ ]:
stim_flash_eeg_sess_epochs = this_sess.get_stim_flash_eeg_epochs(stim)
sess_stim_erp = stim_flash_eeg_sess_epochs[stim].average()

sess_stim_erp_plot = sess_stim_erp.plot_joint()
sess_stim_erp_plot_psd = sess_stim_erp.plot_topomap(times="peaks", ch_type="eeg")
all_sess_stim_erp_plot = sess_stim_erp.compute_psd().plot_topomap()

And here's the average EEG activity for that stimulus across all trials and sessions

In [ ]:
stim_flash_eeg_all_sess_epochs = this_subj.get_stim_flash_eeg_epochs(stim)

all_sess_stim_erp = stim_flash_eeg_all_sess_epochs[stim].average()

all_sess_stim_erp_plot = all_sess_stim_erp.plot_joint()
all_sess_stim_erp_plot_topo = all_sess_stim_erp.plot_topomap(
    times="peaks", ch_type="eeg"
)
all_sess_stim_erp_plot_psd = all_sess_stim_erp.compute_psd().plot_topomap()


We can also get the ERP associated with any recorded EEG events

In [ ]:
response_events = this_subj_behav["solution_key"].unique().tolist()
print(f"{response_events = }")

In [ ]:
selected_events = response_events  # ['trial_start']

with redirect_stdout(open(os.devnull, "w")):
    this_subj_erps = this_subj.get_erp(
        selected_events, tmin=-0.2, tmax=0.5, erp_by_sess=True
    )
# this_subj.get_erp?

all_sess_erp = mne.combine_evoked(list(this_subj_erps.values()), weights="equal")
erp_plot = all_sess_erp.plot()

In [ ]:
this_sess_erp, this_sess_ch_names = (
    this_subj_erps[1].get_data(),
    this_subj_erps[1].info["ch_names"],
)
this_subj_erps[1].plot()

this_sess_erp = pd.DataFrame(this_sess_erp.T, columns=this_sess_ch_names)

px.line(
    this_sess_erp,
    labels={"value": "Amplitude", "index": "Time (ms)", "variable": "Channel"},
    title="ERP Subj 1 - Sess 1",
    width=650,
    height=500,
)

##### Analyze all sessions

In [ ]:
this_sess_res = this_sess.analyze_session(
    save_dir=WD / "analysis/test-export",
    preprocessed_dir=DIRECTORIES.human.prepro,
)

In [ ]:
(
    sess_frps,
    fixation_data,
    eeg_fixation_data,
    gaze_info,
    gaze_target_fixation_sequence,
) = this_sess_res


In [ ]:
all_sess_res = this_subj.analyze_sessions(
    save_dir=WD / "analysis/test-export",
    preprocessed_dir=DIRECTORIES.human.prepro,
    force_preprocess=False,
    reuse_ica=True,
    raise_error=False,
)

In [ ]:
# (
#     sess_frps,
#     fixation_data,
#     eeg_fixation_data,
#     gaze_info,
#     gaze_target_fixation_sequence,
# ) = all_sess_res[5]

In [ ]:
# pprint(sess_frps)
# sess_frps['sequence'][0].plot()
gaze_info
gaze_target_fixation_sequence.groupby("trial_N")["target_ind"].value_counts()
gaze_target_fixation_sequence.query("trial_N==3")
# _fix = fixation_data[1][0]
# plt.plot(_fix[0], _fix[1])

##### Stimuli ERPs

In [ ]:
stim_flash_eeg_eps = this_subj.get_stim_flash_eeg_epochs()
stim_flash_erps = {stim: eps.average() for stim, eps in stim_flash_eeg_eps.items()}

In [ ]:
print(f"Stimuli: {', '.join(stim_flash_erps.keys())}")

In [ ]:
selected_stims = {
    stim: stim_flash_erps[stim] for stim in ["helicopter", "smile", "eye"]
}

with redirect_stdout(open(os.devnull, "w")):
    for i, (stim, erp) in enumerate(selected_stims.items()):
        erp.plot_joint(title=stim)

#### Group level

In [ ]:
human_group.show_dir_struct()

In [ ]:
eeg_metadata = human_group.extract_eeg_metadata()


In [ ]:
combined_stats, acc_fig, rt_fig, scatter = human_group.summarize_behav()

In [ ]:
human_group_behav = human_group.get_behav_data()
human_group_behav["correct"] = human_group_behav["correct"].astype(int)

human_group_acc_stats = human_group_behav.groupby(["sess_N", "subj_N"], observed=False)[
    "correct"
].describe()

display(human_group_acc_stats)


In [ ]:
sns.lineplot(
    data=human_group_behav, x="subj_N", y="correct", hue="sess_N", errorbar=None
)
plt.show()

plots_kwargs = [
    dict(data=human_group_behav, x="sess_N", y="correct"),
    dict(data=human_group_behav, x="sess_N", y="correct", hue="pattern"),
]
for kwargs in plots_kwargs:
    fig, ax = plt.subplots()
    sns.lineplot(ax=ax, **kwargs)
    ax.set_xticks(range(1, 6))
    plt.show()


In [ ]:
human_group_acc_stats.query("subj_N==1")

# LLM Data

### Load Data

In [ ]:
root_data_dir = Path("/Volumes/Realtek 1Tb/PhD Data/experiment1/data")
ann_data_dir = (
    root_data_dir / "ANNs/local_run/sessions-1_to_5-masked_idx(7)-sequence_tokens_acts"
)
ann_export_dir = Path("/Volumes/Realtek 1Tb/PhD Data/experiment1-analysis/ANNs")

qwen2_5_72B = ANNSubjData(
    ann_data_dir, ann_export_dir, ann_id="Qwen--Qwen2.5-72B-Instruct"
)
ann_group = ANNGroupData(ann_data_dir, ann_export_dir)

### Analyze Data

In [ ]:
print("\n".join([i for i in dir(qwen2_5_72B) if not i.startswith("_")]))

In [ ]:
qwen2_5_72B_behav = qwen2_5_72B.get_behav_data()
qwen2_5_72B_behav.head()
# qwen2_5_72B.get_behav_rdms()
# qwen2_5_72B.get_layer_acts()
# qwen2_5_72B.list_contents()
# qwen2_5_72B.get_run_info()
# qwen2_5_72B.get_rdms_on_all_layers()

In [ ]:
ann_group_behav = ann_group.get_behav_data().merge(
    DATASET, on=["item_id", "masked_idx", "pattern", "solution"]
)

# category_cols = {i:"category" for i in (
#     ["ann_id", "pattern", "item_id", "trial_type"]
#     + [f"figure{i}" for i in range(1, 9)]
#     + [f"choice{i}" for i in range(1, 5)]
# )}
# ann_group_behav = ann_group_behav.astype(category_cols)

ann_group_behav.head(3)

In [ ]:
n_items = len(DATASET)
n_item_per_patt = list(set(DATASET.groupby(["pattern"])["item_id"].nunique()))[0]
ann_ids = ann_group_behav["ann_id"].unique()
pattern_index = pd.MultiIndex.from_product(
    [PATTERNS, ann_ids], names=["pattern", "ann_id"]
)

q = "cleaned_response==figure7 & cleaned_response!=solution"

copying_df = (
    ann_group_behav.query(q).groupby(["ann_id"])["pattern"].value_counts().reset_index()
)

copying_df["count"] /= n_item_per_patt
copying_df = copying_df.rename(columns={"count": "pct_copying"})

copying_df = (
    pattern_index.to_frame(index=False)
    .merge(copying_df, on=["pattern", "ann_id"], how="left")
    .fillna(0)
)

copying_df["ann_id"] = copying_df["ann_id"].str.replace(".+--", "", regex=True)
display(copying_df.head(5))


pivot_df = copying_df.pivot(index="pattern", columns="ann_id", values="pct_copying").T
print("Percent of trials where model repeated last visible element, per pattern type:")
pivot_df.style.background_gradient(axis=1, cmap="YlOrRd").format(precision=2)


# Combined Analysis

In [ ]:
combined_data = CombinedData(DIRECTORIES.ann, DIRECTORIES.human, SEQ_FILE)

In [ ]:
print("combined_data available methods/properties:")
print("\n".join([f"\t- {i}" for i in dir(combined_data) if not i.startswith("_")]))

## Compare Performance

In [ ]:
perf_data_all = combined_data.get_perf_data()
print(f"columns: {perf_data_all.columns.tolist()} ")
display(perf_data_all.head())

In [ ]:
perf_data_all.groupby("type")["correct"].describe()
perf_data_all.groupby(["pattern", "type"])["correct"].mean()
# perf_data_all.groupby("id")['correct'].mean()

## Representational Similarity Analysis

In [ ]:
human_ds, llm_ds = combined_data.get_rsa_datasets(
    human_data_dir=DIRECTORIES.human.analyzed / "RSA-FRP-frontal",
    ann_data_dir=DIRECTORIES.ann.analyzed
    / "RSA-seq_tokens-metric_correlation/best_layer",
    level="pattern",
)

In [ ]:
rsa_repr = combined_data.compare_representations(
    human_ds,
    llm_ds,
    n_perm=0,
    n_boot=0,
    boot_conf_int=(2.5, 97.5),
    random_state=None,
    pbar=True,
    pbar_perm=True,
    pbar_boot=True,
    descriptor_match=None,
    similarity_metric="corr",
    dissimilarity_metric="correlation",
    tail="two-sided",
    save_dir=WD / "TEST",
)

observed_corrs, permuted_corrs, bootstrap_corrs, df_res = rsa_repr


In [ ]:
df_res.query("id1=='group'")

In [ ]:
df_res["corr"].plot(kind="hist")

In [ ]:
res = []
res_dfs = []
for _human_dir in [
    "RSA-FRP-frontal",
    "RSA-Response_ERP-frontal",
    "RSA-Rest_ERP-frontal",
]:
    human_dir = DIRECTORIES.human.analyzed / _human_dir

    human_ds, llm_ds = combined_data.get_rsa_datasets(
        human_data_dir=human_dir,
        ann_data_dir=DIRECTORIES.ann.analyzed
        / "RSA-seq_tokens-metric_correlation/best_layer",
        level="pattern",
    )

    rsa_repr = combined_data.compare_representations(
        human_ds,
        llm_ds,
        n_perm=0,
        n_boot=0,
        boot_conf_int=(2.5, 97.5),
        random_state=None,
        pbar=True,
        pbar_perm=True,
        pbar_boot=True,
        descriptor_match=None,
        similarity_metric="corr",
        dissimilarity_metric="correlation",
        tail="two-sided",
        save_dir=WD / "TEST",
    )

    observed_corrs, permuted_corrs, bootstrap_corrs, df_res = rsa_repr
    df_res["corr"].plot(kind="hist")
    plt.show()

    df_res = df_res.query("id1=='group'").copy()
    df_res["type"] = _human_dir.split("-")[1]
    res_dfs.append(df_res)

df_res = pd.concat(res_dfs).reset_index(drop=True)

In [ ]:
df_res["id2"] = df_res["id2"].replace(ANN_ID_MAPPING)

fig, ax = plt.subplots()
sns.barplot(df_res, x="corr", y="id2", hue="type", order=ANN_ID_ORDER, ax=ax)
ax.grid(axis="x", ls="--", lw=0.75)
ax.legend(bbox_to_anchor=(1, 1))
plt.show()


# DRAFT

## TESTING

In [ ]:
human_group.get_frp_data()

## Controls

### Control 1 - EEG averaging influence

In [ ]:
import numpy as np
from tqdm.auto import tqdm
from utils.analysis_utils import save_pickle, read_file
from mne.io import concatenate_raws
from rsatoolbox.data import Dataset
from rsatoolbox.rdm import calc_rdm, compare_cosine, compare as compare_rdm
from analysis_rsa import get_reference_rdms, get_ds_and_rdm
from rsatoolbox.rdm.rdms import load_rdm
from rsatoolbox.data.dataset import load_dataset
import re

In [ ]:
save_dir = WD / "analysis/TEST/RSA-Random_EEG_Windows"
save_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# for subj_N, subj_obj in tqdm(human_group.subjects.items()):
#     behav, et_trials, eeg_trials = subj_obj.get_trials_data(DIRECTORIES.human.prepro)
#     behav.to_csv(save_dir / f"subj{subj_N:02}-behav.csv")

#     et_trials_concat = mne.concatenate_raws(et_trials)
#     et_trials_concat.save(save_dir / f"{subj_N:02}-trials_et.fif")

#     eeg_trials_concat = mne.concatenate_raws(eeg_trials)
#     eeg_trials_concat.save(save_dir / f"{subj_N:02}-trials_eeg.fif")


In [ ]:
_files = [f for f in save_dir.glob("*") if f.is_file() and not f.name.startswith(".")]
existing = []
subj_Ns = [int(re.search(r"subj_(\d{2})", f.name).groups()[0]) for f in _files]
for subj_N in set(subj_Ns):
    if subj_Ns.count(subj_N) == 4:
        existing.append(subj_N)
print(existing)

In [ ]:
fixation_estmd_duration = 0.600
eeg_sfreq = 2048
window_duration = round(fixation_estmd_duration * eeg_sfreq)


def save_ds_and_rdm(ds, rdm, level, save_dir=save_dir):
    base_fname = f"human-subj_{ds.descriptors['subj_N']:02}-{level}_lvl.hdf5"
    ds.save(save_dir / f"dataset-{base_fname}", file_type="hdf5", overwrite=True)
    rdm.save(save_dir / f"rdm-{base_fname}", file_type="hdf5", overwrite=True)


for subj_N, subj_obj in tqdm(human_group.subjects.items()):
    if subj_N in existing:
        continue

    behav, et_trials, eeg_trials = subj_obj.get_trials_data(DIRECTORIES.human.prepro)
    info = eeg_trials[0].info
    montage = eeg_trials[0].get_montage()

    processed_eeg_trials = []

    for eeg_trial in tqdm(eeg_trials):
        eeg_trial_arr = eeg_trial.get_data()

        n_samples = int(eeg_trial_arr.shape[1] / window_duration)

        eeg_trial_windows = np.array_split(eeg_trial_arr, n_samples, axis=1)

        duration = min([e.shape[-1] for e in eeg_trial_windows])

        eeg_trial_windows = [e[:, :duration] for e in eeg_trial_windows]

        selected_window_inds = sorted(
            np.random.choice(
                len(eeg_trial_windows),
                size=np.random.randint(3, round(n_samples * 0.9)),
                replace=False,
            )
        )

        selected_eeg_trial_windows = [
            eeg_trial_windows[i] for i in selected_window_inds
        ]

        eeg_trial_windows_avg = np.stack(selected_eeg_trial_windows).mean(axis=0)
        eeg_trial_windows_avg = eeg_trial_windows_avg[np.newaxis, :, :window_duration]

        eeg_trial_windows_avg_mne = mne.EpochsArray(
            eeg_trial_windows_avg, info, verbose=False
        )
        eeg_trial_windows_avg_mne.set_montage(montage)
        # eeg_trial_windows_avg_plot = eeg_trial_windows_avg_mne.average().plot()
        processed_eeg_trials.append(eeg_trial_windows_avg_mne)

    # * Sequence/Item-level
    obs_descriptors = behav["pattern"].to_numpy()
    measurements = [
        i.get_data(picks=c.EEG_CHAN_GROUPS.frontal).squeeze().flatten()
        for i in processed_eeg_trials
    ]

    measurements = np.stack(measurements)

    ds = Dataset(
        measurements=measurements,
        descriptors={"subj_N": subj_N},
        obs_descriptors={"patterns": obs_descriptors},
    )
    rdm = calc_rdm(ds, "correlation")
    save_ds_and_rdm(ds, rdm, level="sequence", save_dir=save_dir)

    # * Pattern-level
    measurements = [
        i.get_data(picks=c.EEG_CHAN_GROUPS.frontal).squeeze().flatten()
        for i in processed_eeg_trials
    ]
    measurements = np.stack(measurements)

    pattern_inds = behav.groupby("pattern").groups
    pattern_inds = {p: pattern_inds[p] for p in PATTERNS}
    measurements = np.stack(
        [measurements[inds].mean(axis=0) for inds in pattern_inds.values()]
    )

    ds = Dataset(
        measurements=measurements,
        descriptors={"subj_N": subj_N},
        obs_descriptors={"patterns": PATTERNS},
    )
    rdm = calc_rdm(ds, "correlation")
    save_ds_and_rdm(ds, rdm, level="pattern", save_dir=save_dir)

In [ ]:
# subj_obj = human_group.subjects[1]
# subj_N = subj_obj.subj_N
# # behav, et_trials, eeg_trials = subj_obj.get_trials_data(DIRECTORIES.human.prepro)

In [ ]:
# get_ds_and_rdm(
#     measurements: np.ndarray,
#     dissimilarity_metric: str,
#     ds_fpath: Optional[Path] = None,
#     rdm_fpath: Optional[Path] = None,
#     ds_type: str = "regular",
#     descriptors: Optional[Dict] = None,
#     obs_descriptors: Optional[Dict] = None,
#     channel_descriptors: Optional[Dict] = None,
#     time_descriptors: Optional[Dict] = None,
# )

In [ ]:
plt.imshow(human_random_segmt_rdm_patt_level.get_matrices()[0])

In [ ]:
llm_rdms = []
corr_vals = []
for llm, ds in llm_ds.items():
    llm_rdm = calc_rdm(ds, "correlation")
    # plt.imshow(llm_rdm.get_matrices()[0])
    # plt.show()
    comp = compare_rdm(llm_rdm, human_random_segmt_rdm_patt_level, method="corr")
    corr_vals.append(comp[0])
corr_vals

### Control 2 - 